# XiangqiRL — PPO training on Kaggle

Trains the policy-value network with PPO against the random and heuristic baselines (Phase 4).

**Before running**
1. Notebook settings: **Accelerator = GPU T4** (CPU works too, only slower) and **Internet = On** (needed to `git clone` and `pip install gymnasium`).
2. The project is cloned from GitHub in section 1 (`REPO_URL`). A public repo is cloned anonymously. For a **private** repo, add a Kaggle *Secret* named `GITHUB_TOKEN` (Add-ons → Secrets) with a personal access token that has `repo` read access.
3. *(Optional, to continue an earlier run)* upload `latest.pt` from a previous session as a dataset and set `RESUME_FROM` below.

Passing tests shows the code behaves as specified; it does **not** by itself show that the agent plays well. Judge that from the evaluation section at the end.

In [ ]:
import os, shutil, subprocess, sys, glob, time
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Clone and install the project

In [ ]:
REPO_URL = "https://github.com/TamasTran/xiangqi-rl.git"
BRANCH   = "main"          # branch, tag or commit to use
PROJECT  = "/kaggle/working/xiangqi-rl"

import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError as exc:
    raise RuntimeError(
        "Cannot reach github.com -> turn Internet ON in the notebook settings "
        "(right sidebar > Session options > Internet), then re-run."
    ) from exc

def try_clone(url: str) -> subprocess.CompletedProcess:
    if os.path.exists(PROJECT):
        shutil.rmtree(PROJECT)
    return subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, url, PROJECT],
        capture_output=True, text=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )

# 1) anonymous clone (works for a public repo); a stale token must not break this.
r = try_clone(REPO_URL)
if r.returncode != 0:
    # 2) private repo: use the Kaggle Secret GITHUB_TOKEN if one exists.
    print("anonymous clone failed:", r.stderr.strip()[-400:])
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Clone failed and no GITHUB_TOKEN secret is available. For a private repo add a "
            "Kaggle Secret named GITHUB_TOKEN (Add-ons > Secrets) and attach it to this notebook."
        ) from exc
    authed = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    r = try_clone(authed)
    if r.returncode != 0:
        raise RuntimeError("authenticated clone failed:\n" + r.stderr.replace(token, "***")[-600:])
    subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", REPO_URL], check=True)  # drop token

os.chdir(PROJECT)
print("commit:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# --no-deps keeps Kaggle's preinstalled (CUDA) torch; only the small extras are installed.
!pip install -q -e . --no-deps
!pip install -q gymnasium pyyaml

# The kernel is already running, so the editable install is not visible to *this* process
# until its path is refreshed (new `!python ...` processes see it without this).
import importlib
sys.path.insert(0, os.path.join(PROJECT, "src"))
importlib.invalidate_caches()

import xiangqi_rl, gymnasium, yaml
print("xiangqi_rl", xiangqi_rl.__version__, "| gymnasium", gymnasium.__version__)

## 2. Tests and smoke test (about a minute)

In [ ]:
!python -m pip install -q pytest
!python -m pytest -q -x tests

In [ ]:
!python scripts/train_ppo.py --config configs/ppo_baseline.yaml --smoke-test

## 3. Training configuration

Edit these values. Everything else comes from `configs/ppo_baseline.yaml`.
One iteration of the baseline config (2048 decisions) took about 25 s on a CPU-only
development machine; the game simulation is CPU-bound, so a T4 mainly speeds up the update.

In [ ]:
RUN_NAME     = "run1"                    # change for every NEW run (logs/checkpoints are per run)
OUTPUT_DIR   = f"/kaggle/working/runs/{RUN_NAME}"
ITERATIONS   = 100                       # total rollout+update cycles
ROLLOUT_STEPS, MINIBATCH = 2048, 256
LEARNING_RATE = 3e-4                     # lower (e.g. 1e-4) if approx_kl / clip_fraction stay high
EVAL_INTERVAL, EVAL_GAMES = 10, 20       # evaluate every N iterations, games per opponent (even)
SEED = 0
RESUME_FROM = None                       # continue a run: f"{OUTPUT_DIR}/latest.pt" or "/kaggle/input/<dataset>/latest.pt"

overrides = {
    "total_iterations": ITERATIONS, "rollout_steps": ROLLOUT_STEPS, "minibatch_size": MINIBATCH,
    "learning_rate": LEARNING_RATE, "eval_interval": EVAL_INTERVAL, "eval_games": EVAL_GAMES,
    "seed": SEED, "output_dir": OUTPUT_DIR,
}
SET_ARGS = " ".join(f"--set {k}={v}" for k, v in overrides.items())
RESUME_ARGS = f"--resume {RESUME_FROM}" if RESUME_FROM else ""
print(SET_ARGS, RESUME_ARGS)

## 4. Train

A new run needs a new `RUN_NAME` (the trainer refuses to mix two runs in one directory). To continue a run, keep `RUN_NAME` and set `RESUME_FROM = f"{OUTPUT_DIR}/latest.pt"`; raise `ITERATIONS` to train longer.

In [ ]:
!python scripts/train_ppo.py --config configs/ppo_baseline.yaml --device auto {SET_ARGS} {RESUME_ARGS}

## 5. Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv(f"{OUTPUT_DIR}/train_log.csv")
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.flat, ["episode_reward", "policy_loss", "value_loss",
                               "entropy", "approx_kl", "explained_variance"]):
    ax.plot(log["iteration"], log[col]); ax.set_title(col); ax.set_xlabel("iteration"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
log[["iteration", "decisions", "episode_reward", "wins", "draws", "losses",
     "env_seconds", "inference_seconds", "update_seconds"]].tail()

In [ ]:
ev = pd.read_csv(f"{OUTPUT_DIR}/eval_log.csv")
ev[["iteration", "opponent", "games", "wins", "draws", "losses", "win_rate", "score",
    "average_plies", "invalid_actions"]]

## 6. Final evaluation (separate games, many of them)

20 games per opponent is noisy. Use at least a few hundred before drawing conclusions,
and compare checkpoints with the same seed.

In [ ]:
!python scripts/evaluate_ppo.py --checkpoint {OUTPUT_DIR}/best.pt --games 200 --seed 12345 --device auto

## 7. Package the results

Download `ppo_run.zip` from the *Output* tab (or save a notebook version) and upload `latest.pt` as a dataset to resume later.

In [ ]:
shutil.make_archive("/kaggle/working/ppo_run", "zip", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))